# Workflow Optimizer — V0

**What this does.** You describe a task (e.g. *"grade-school math word problems, integer answers"*). The notebook then has an AI agent **design candidate solution strategies and iteratively make them cheaper** without losing accuracy, runs them on real examples, and lets you pick the strategy with the accuracy/cost tradeoff you want.

**The key idea: a "workflow" is a small Python function.** Every candidate strategy is written as a function with this exact shape:

```python
def solve(question, call_model):
    # ...any Python you want...
    return answer
```

- `question` — one input from the dataset (a string).
- `call_model` — **a function the notebook hands to your program.** Calling `call_model("some prompt")` sends that prompt to a model and returns the model's reply as a string. It is *not* a string or a model name; it's the single gateway to the model (defined in the *"one model call"* cell below). Everything a workflow does with a model goes through it — which is how the notebook meters cost and enforces a budget.
- `answer` — whatever the strategy produced; the notebook then extracts and grades it.

Because `solve` is ordinary code, it can express **any** strategy — a single `call_model(...)` call, chain-of-thought, sampling 5 times and majority-voting, breaking the problem into steps, or routing easy questions to a cheap model and hard ones to an expensive one. Nothing is hard-coded; the agent writes the code.

**The pipeline (this notebook, top to bottom):**
1. **Analyze the task** — a model reads your description and works out how answers should be *extracted* and *checked*, and generates example data if you didn't supply any.
2. **Optimize (a loop)** — a Claude Code agent designs candidate `solve` programs, then **iterates**: each round it sees the best workflows so far and tries to make *cheaper* ones that keep accuracy, building up an archive of candidates.
3. **Choose** — rank the surviving tradeoffs on a held-out test split, then pick the methodology you want (each is a real point on the accuracy/cost **Pareto frontier** — nothing else beats it on *both* price and accuracy at once).

**Setup.** Every code cell makes real Anthropic API calls, so set `ANTHROPIC_API_KEY` first. Each workflow runs under a per-query budget (a cap on model calls + tokens); a program that overspends or crashes simply scores 0.

## Setup — the model catalog, cost, and client

Everything model-related comes from one place: a small **`Model`** record per model (id, input/output price, and whether it supports `effort`/thinking), collected in **`MODEL_SPECS`**. From that list we derive `MODELS` (the ids, cheapest → most expensive — the search pool, and what a workflow routes over) and `BY_ID` (id → `Model`). `cost_usd(model, usage)` and the call layer look a model up in `BY_ID`, so cost and capabilities can't drift from the list. `cost_usd` is **cache-aware** (writes 1.25×, reads 0.10× the input rate). `client` is the Anthropic client every call goes through.

In [ ]:
import os, re, json, statistics
from dataclasses import dataclass
from typing import Literal

import anthropic
from pydantic import BaseModel, ConfigDict
import pandas as pd
import matplotlib.pyplot as plt

# ---- The models the optimizer can use -------------------------------------
# One record per model; everything downstream (cost, the effort/thinking guard,
# the search pool) reads from this single list, so nothing drifts out of sync.
@dataclass
class Model:
    id: str            # the API model id
    price_in: float    # USD per 1,000,000 input tokens
    price_out: float   # USD per 1,000,000 output tokens
    thinks: bool       # supports the effort / adaptive-thinking params

MODEL_SPECS = [
    Model("claude-haiku-4-5", 1.0,  5.0, thinks=False),
    Model("claude-sonnet-5",  3.0, 15.0, thinks=True),
    Model("claude-opus-4-8",  5.0, 25.0, thinks=True),
]
BY_ID = {m.id: m for m in MODEL_SPECS}      # id -> Model, for cost + capability lookups
MODELS = [m.id for m in MODEL_SPECS]        # ids, cheapest -> most expensive (the search pool)

# ---- Cost of one call, from its token usage -------------------------------
# Prompt-cache multipliers on the INPUT rate: a cache write costs a bit more once
# (1.25x), a read is ~90% off (0.10x).
CACHE_WRITE_MULT = 1.25
CACHE_READ_MULT = 0.10

def cost_usd(model, usage):
    # usage: {"input", "output", "cache_write", "cache_read"} token counts.
    spec = BY_ID[model]
    tokens = (usage["input"]        * spec.price_in
              + usage["cache_write"] * spec.price_in * CACHE_WRITE_MULT
              + usage["cache_read"]  * spec.price_in * CACHE_READ_MULT
              + usage["output"]      * spec.price_out)
    return tokens / 1_000_000

# ---- Anthropic API client -------------------------------------------------
# Every rollout is a real API call, so a key is required.
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError("Set ANTHROPIC_API_KEY in your shell env before running.")
client = anthropic.Anthropic()

print("Models:", ", ".join(MODELS))
print("Anthropic client ready.")

## Define the task — *the only cell you edit per task*

`SEED_PROMPT` describes the task in plain English. `DATASET` is optional — your own labeled examples as a list of `{"question": ..., "answer": ...}`; leave it `None` and the notebook generates some. Everything below adapts to whatever you put here.

In [ ]:
# ---- Define the problem (the ONLY per-task input) -------------------------
# Describe the task in SEED_PROMPT. Optionally supply your own labeled DATASET
# as a list of {"question": <input str>, "answer": <target>}; leave it None to
# have one generated. The answer format (numeric / exact-match / free-form) is
# figured out automatically in the analysis step below.
SEED_PROMPT = ("Build a system that generates clinical notes for a given doctor-patient conversation")
DATASET = None

# When DATASET is None, how many labeled examples to generate. Bigger = less noisy
# accuracy + a finer Pareto frontier, but proportionally more API calls in the search.
N_DATASET = 100

## The one model call — what `call_model` actually is

This defines the single function that reaches a model. The `call_model` your `solve(question, call_model)` receives **is this function** (wrapped with cost metering — see *The runtime* cell later). You call it as `call_model(prompt)` and get back a `Reply` — a string carrying the full response (see *The runtime* cell).

A workflow can also pass:
- `model=` — route this call to a specific model.
- `system=` — a system prompt for this call.
- `tools=["code_execution"]` and/or `["web_search"]` — let the model run Python or search the web (these run server-side; results come back in the reply).
- `effort="low"…"max"` — let a large model think step-by-step before answering.
- `schema=` — a JSON Schema the reply must match; `reply.data` is the parsed object. It constrains only the text the model writes at the end, so it composes with `tools=`.

There is no per-call output cap to tune: every call uses one generous `MAX_OUTPUT_TOKENS` ceiling. A cap wouldn't save money (you pay for the tokens a reply actually uses) and a tight one only risks cutting an answer short — spend is controlled by the per-query budget in `Runtime`.

Routing *every* model call through this one place is what lets the notebook measure cost and enforce a per-query budget, no matter what the program does. (`_call_api` is the raw call; `Runtime.call_model`, defined later, wraps it with metering and is what a program actually receives.)

**Prompt caching.** The prompt and system prompt carry a cache breakpoint, so resending the *same* prompt to the *same* model reads it from cache — the reused tokens bill at ~10% of the input rate instead of full price (the first send pays a one-time 25% write surcharge). The cache is keyed by model, so the *same* prompt to a *different* model is a fresh miss with no discount. Only prompts above the model's minimum (~1–2k tokens) cache at all; shorter ones bill normally. `_call_api` returns `(final_text, blocks, usage)`, with `usage` split into `input` / `output` / `cache_write` / `cache_read`, and `cost_usd` prices each accordingly.

In [ ]:
# ---- One model call (the only way a workflow reaches a model) --------------
# A program calls call_model(...) with any of these; they map straight onto the API here.
# Server-side tools (code execution, web search) run on Anthropic's side, so
# there's nothing to execute locally — the model uses them and the results come
# back in the same reply. `effort` turns on the model's own step-by-step thinking.
# `schema` constrains the reply to JSON matching it.
#
# The prompt (and system) carry a cache breakpoint, so resending the SAME prompt to
# the SAME model reads it from cache instead of paying full price. The cache is
# server-side and keyed by model, so a different model is always a fresh miss.

TOOL_DEFS = {   # name a program passes -> the tool definition sent to the API
    "code_execution": {"type": "code_execution_20260521", "name": "code_execution"},
    # allowed_callers=["direct"] is REQUIRED for the cheap model: the _20260209 web
    # tools default to being called from inside code execution (that's how dynamic
    # filtering works), and haiku can't do programmatic tool calling — without this
    # every web_search call on the default model dies with a 400.
    "web_search":     {"type": "web_search_20260209", "name": "web_search",
                       "allowed_callers": ["direct"]},
}

MAX_TOOL_TURNS = 5   # cap on API calls while a server-side tool keeps pausing the turn

# One output ceiling for every call, set high enough to never be the reason an
# answer is short. This is NOT a cost knob — you are billed for the tokens a reply
# actually uses, so a tight cap saves nothing. It has to be generous because
# THINKING COUNTS AGAINST IT: at effort="max" a 16k ceiling was measured burning
# the entire budget on thinking and returning an empty answer. Cost is controlled
# by the per-query budget in Runtime, which is what a workflow is measured against.
MAX_OUTPUT_TOKENS = 64000

def _get_usage(message):
    # Anthropic returns these token counts in the response's usage block; split
    # so cost_usd can price cached vs. fresh tokens differently.
    return {
        "input":       message.usage.input_tokens,
        "output":      message.usage.output_tokens,
        "cache_write": getattr(message.usage, "cache_creation_input_tokens", 0) or 0,
        "cache_read":  getattr(message.usage, "cache_read_input_tokens", 0) or 0,
    }

def _call_api(model, prompt, system=None, tools=None, effort=None, schema=None):
    # Returns (final_text, blocks, usage). `blocks` is every content block from every
    # API response this call made — tool calls, tool results, text — kept for the trace.
    # A cache breakpoint on the prompt -> identical resends to this model are cheap.
    content = [{"type": "text", "text": str(prompt),
                "cache_control": {"type": "ephemeral"}}] # cache_control ephemeral means auto-expiring cache (typically 5m)
    request = {
        "model": model,
        "max_tokens": MAX_OUTPUT_TOKENS,
        "messages": [{"role": "user", "content": content}],
    }
    if system:
        request["system"] = [{"type": "text", "text": system,
                              "cache_control": {"type": "ephemeral"}}]
    if tools:
        request["tools"] = []
        for name in tools:
            request["tools"].append(TOOL_DEFS[name])

    # effort and schema share one output_config, so build it before assigning.
    output_config = {}
    if effort and model in BY_ID and BY_ID[model].thinks:
        request["thinking"] = {"type": "adaptive"}
        output_config["effort"] = effort
    else:
        request["thinking"] = {"type": "disabled"}        # default: the strategy is the only knob
    if output_config:
        request["output_config"] = output_config
    if schema is not None:
        # Constrains ONLY the text the model writes at the end — tool calls and
        # tool results in the same reply are untouched. Takes a Pydantic model
        # class (what this notebook uses) or a raw JSON Schema dict (what a
        # workflow program passes, since the sandbox has no pydantic).
        request["output_format"] = (
            schema if isinstance(schema, type) and issubclass(schema, BaseModel)
            else {"type": "json_schema", "schema": schema})

    turn_texts = []                        # text of each API response, kept separate
    blocks = []                            # every content block, in order
    usage = {"input": 0, "output": 0, "cache_write": 0, "cache_read": 0}
    turns = 0
    while turns < MAX_TOOL_TURNS:          # resume the turn while a tool keeps pausing it
        turns += 1
        # Streamed because MAX_OUTPUT_TOKENS is large: the SDK refuses a
        # non-streaming request whose ceiling could outlive the HTTP timeout.
        # get_final_message() reassembles the same Message a create() would return.
        with client.messages.stream(**request) as stream:
            message = stream.get_final_message()
        got = _get_usage(message)

        # hardcoded usage for Anthropic
        usage["input"]       += got["input"]
        usage["output"]      += got["output"]
        usage["cache_write"] += got["cache_write"]
        usage["cache_read"]  += got["cache_read"]

        blocks.extend(message.content)
        # Join the text blocks WITHIN this response (citations split one message
        # into several), but keep responses apart — see below.
        turn_texts.append("".join(b.text for b in message.content if b.type == "text"))

        if message.stop_reason != "pause_turn":
            break

        request["messages"].append({"role": "assistant", "content": message.content})

    # The LAST response holds the answer; earlier ones are the model working up to
    # it. Concatenating across responses would splice that preamble onto the answer
    # — and with `schema` set it would produce unparseable JSON.
    final_text = turn_texts[-1] if turn_texts else ""
    return final_text, blocks, usage

**Examples — pricing + the one model call.** `_call_api` returns `(final_text, blocks, usage)` — `blocks` is every content block from every API response the call made (tool calls, tool results, text), and `usage` counts `input` / `output` / `cache_write` / `cache_read` tokens; `cost_usd(model, usage)` prices them. Cache reads bill ~10% of the input rate, so resending the same prompt to the same model is cheap.

```python
# haiku input is $1/M. The same 2,000-token prompt, three ways:
cost_usd("claude-haiku-4-5", {"input": 2000, "output": 0, "cache_write": 0, "cache_read": 0})  # uncached   -> 0.0020
cost_usd("claude-haiku-4-5", {"input": 0, "output": 0, "cache_write": 2000, "cache_read": 0})  # first send -> 0.0025
cost_usd("claude-haiku-4-5", {"input": 0, "output": 0, "cache_write": 0, "cache_read": 2000})  # sent again -> 0.0002

# _call_api returns the split usage (this short prompt is below the cache minimum, so no caching):
_call_api("claude-haiku-4-5", "What is 2+2?")   # -> ("4", [TextBlock(...)], {"input": 14, "output": 3, "cache_write": 0, "cache_read": 0})
_call_api("claude-sonnet-5", "Solve it.", system="You are a careful math tutor.")
_call_api("claude-sonnet-5", "Compute 47 * 53.", tools=["code_execution"])
_call_api("claude-opus-4-8", "A tricky word problem...", effort="high")
_call_api("claude-haiku-4-5", "Name a color.", schema={"type": "object", "properties": {"answer": {"type": "string"}}, "required": ["answer"], "additionalProperties": False})
```

## Grading an answer

A workflow **returns its answer** — `solve()` hands back the value to grade, so there is no extraction step. `check_answer(prediction, gold, check_type)` returns a **score in [0, 1]**, and a program's accuracy is the **mean score** over the dataset.

- **Nothing is parsed out of prose, on either side.** `numeric` requires the answer to *be* a number (`as_number`): `"42"` scores, `"42 apples"` does not. Searching prose for a number instead would grade `"42 out of 100"` as `100`. `exact` compares case-insensitively after stripping.
- **The judge returns a number, not a sentence containing one.** `llm_judge` is a **graded** score from `judge_score` — a cheap model scores the candidate 0–100 under `JUDGE_SCHEMA` against a **task-specific rubric** the analyzer writes and validates (generic-rubric fallback if it doesn't discriminate). So for free-form tasks, "accuracy" is *mean quality*, not pass/fail.
- A program may return the answer directly, a dict `{"answer": ..., ...}`, or a schema-constrained `Reply` as-is. A returned object with no `answer` key raises, and the error lands in the record — see `final_answer` in the runtime cell.

`extract_last_number` survives only as a **helper handed to workflow programs**, for pulling a value out of a free-text intermediate result mid-pipeline. Grading no longer uses it.

In [ ]:
# ---- Grade an answer against the gold answer -------------------------------
# check_answer returns a SCORE in [0, 1]: 1.0 / 0.0 for numeric & exact, and a
# graded quality score for llm_judge (see judge_score). The analyzer (next cell)
# picks check_type and, for llm_judge, writes + validates JUDGE_RUBRIC.
#
# Nothing is extracted anywhere in grading: a workflow RETURNS its answer, and the
# judge is asked for a schema-constrained score, so neither side parses prose.

JUDGE_MODEL = "claude-haiku-4-5"   # grades free-form answers; bump to sonnet for a stronger judge
JUDGE_RUBRIC = ""                  # task-specific rubric, set by the analyzer for llm_judge tasks
JUDGE_TASK = ""                    # the task description, set by the analyzer

class JudgeScore(BaseModel):
    """The judge replies with a number, not a sentence containing one."""
    model_config = ConfigDict(extra="forbid")
    score: int

def extract_last_number(text):
    # The last number in the text, or None. NOT used by grading — it is handed to
    # workflow programs, which may still want to pull a value out of a free-text
    # intermediate result mid-pipeline.
    numbers = re.findall(r"-?\d[\d,]*\.?\d*", text or "")
    if not numbers:
        return None
    try:
        return float(numbers[-1].replace(",", ""))
    except ValueError:
        return None

def as_number(value):
    # A numeric answer must BE a number. A program returns its answer, so there is
    # nothing to search for: "42" parses, "42 apples" does not. Searching prose for
    # the last number instead would grade "42 out of 100" as 100.
    try:
        return float(str(value).replace(",", "").strip())
    except ValueError:
        return None

def judge_score(prediction, gold, rubric="", task=""):
    # Grade a free-form candidate from 0 to 1 against the task rubric (the gold is
    # an example of a good answer, not the only acceptable one). 0.0 if the judge
    # refuses or its reply doesn't parse.
    criteria = rubric.strip() or "Does the candidate correctly and completely satisfy the task?"
    prompt = (f"Task: {task}\n\nGrading rubric:\n{criteria}\n\n"
              f"Reference (an example of a good answer):\n{gold}\n\n"
              f"Candidate answer:\n{prediction}\n\n"
              "Score how well the candidate satisfies the rubric for the task, from 0 to 100 "
              "(100 = fully correct/complete, 0 = wrong or empty).")
    reply, _, _ = _call_api(JUDGE_MODEL, prompt, schema=JudgeScore)
    try:
        score = JudgeScore.model_validate_json(reply).score
    except ValueError:                         # refusal, or a reply past the ceiling
        return 0.0
    return max(0.0, min(1.0, score / 100.0))   # a schema can't bound a range, so clamp

def check_answer(prediction, gold, check_type):
    if check_type == "numeric":
        p, g = as_number(prediction), as_number(gold)
        return 1.0 if (p is not None and g is not None and abs(p - g) < 1e-6) else 0.0
    if check_type == "exact":
        return 1.0 if str(prediction).strip().casefold() == str(gold).strip().casefold() else 0.0
    # "llm_judge": a graded 0-1 quality score against the task rubric.
    return judge_score(prediction, gold, JUDGE_RUBRIC, JUDGE_TASK)

**Examples — grading.** `check_answer` returns a **score in [0, 1]**.

```python
as_number("42")          # -> 42.0
as_number(" 1,024 ")     # -> 1024.0   (thousands separators are fine)
as_number("42 apples")   # -> None     (an answer must BE the number)

check_answer("42", 42, "numeric")           # -> 1.0
check_answer("42 apples", "42", "numeric")  # -> 0.0   (the program broke the contract)
check_answer("Positive", "positive", "exact")  # -> 1.0   (case-insensitive)
check_answer("cat", "dog", "exact")            # -> 0.0

# llm_judge grades 0-1 against the task rubric (hits the API, replies under JUDGE_SCHEMA):
judge_score("A concise clinical note...", gold_note, JUDGE_RUBRIC, JUDGE_TASK)  # -> e.g. 0.8

# still available to workflow PROGRAMS for their own intermediate parsing:
extract_last_number("Sold 75 of 84 apples, so 9 remain.")   # -> 9.0
```

## Step 1 · Analyze the task — infer format, make data

One structured model call reads your `SEED_PROMPT` and fills in a **`TaskProfile`** — a Pydantic model, so one definition both constrains the reply and types the object we read back. It holds only what is actually consumed downstream:

| field | what reads it |
| --- | --- |
| `description` | the design agent's brief, the judge's task, the dataset generator |
| `check_type` | picks the grader (numeric / exact / llm_judge) |
| `judge_rubric` | `build_judge`, for free-form tasks only |
| `answer_examples` | validates the rubric, and shows the designer the target answer format |

`build_judge` checks the rubric scores a gold answer high and an empty answer low, falling back to a generic judge if it doesn't discriminate.

If you gave no `DATASET`, it also generates **`N_DATASET`** labeled examples — in **small batches** (one giant call would run past `MAX_OUTPUT_TOKENS` and return truncated JSON), spread across model-planned **facets** and deduped on a normalized key (plus a per-batch avoid-list) so batches don't re-cover the same cases. The data is split into **DEV** (which the design agent may test against) and a held-out **TEST** set used only for the final ranking.

In [ ]:
# ---- Analyze the task: infer format + judge rubric, get data ---------------
# The analyzer's outputs are Pydantic models rather than hand-written JSON Schema
# dicts: one definition both constrains the model's reply and types the object we
# read back, so the two can't drift. Every field here is consumed downstream —
# if something is only printed, it doesn't belong in the profile.
ANALYSIS_MODEL = "claude-opus-4-8"

class TaskProfile(BaseModel):
    """What the analyzer infers about your task."""
    model_config = ConfigDict(extra="forbid")   # -> additionalProperties: false

    description: str                            # briefs the design agent; grounds the judge
    check_type: Literal["numeric", "exact", "llm_judge"]   # picks the grader
    judge_rubric: str                           # llm_judge only; "" otherwise
    answer_examples: list[str]                  # validates the rubric; shown to the designer

class Example(BaseModel):
    model_config = ConfigDict(extra="forbid")
    question: str
    answer: str

class Dataset(BaseModel):
    model_config = ConfigDict(extra="forbid")
    data: list[Example]

class Facets(BaseModel):
    model_config = ConfigDict(extra="forbid")
    facets: list[str]

def generate_json(model, prompt, schema_model):
    # One structured-output call, validated straight into `schema_model`. The reply
    # is guaranteed to match it as long as it fits in MAX_OUTPUT_TOKENS — a reply
    # cut off at the ceiling is invalid JSON, so keep expected output well under it.
    text, _, _ = _call_api(model, prompt, schema=schema_model)
    return schema_model.model_validate_json(text)

def analyze_task(seed_prompt, dataset):
    examples = ""
    if dataset:
        examples = "\n\nExamples (input -> answer):\n"
        for item in dataset[:5]:
            examples += f"- {item['question']} -> {item['answer']}\n"
    prompt = (
        f"A user wants to build an LLM workflow for this task:\n{seed_prompt}{examples}\n\n"
        "Reply with:\n"
        "- description: one clear paragraph describing the task for a workflow designer.\n"
        "- check_type: how to score a prediction against the gold answer — 'numeric' for "
        "numbers, 'exact' for short labels/strings, 'llm_judge' for free-form/semantic answers.\n"
        "- judge_rubric: ONLY for check_type 'llm_judge' — a short rubric (2-5 concrete criteria) "
        "defining what makes a candidate output correct / high-quality for THIS task; it grades "
        "candidates from 0 to 100. Return an empty string for other check_types.\n"
        "- answer_examples: 2-4 examples of CORRECT final answers for this task, in exactly the "
        "form a workflow should return them — a BARE value for numeric/exact tasks (the number "
        "or the label alone, no words or units around it), a full reference answer for free-form "
        "ones. These are shown to the workflow designer as the target format, and are used to "
        "sanity-check the judge.")
    return generate_json(ANALYSIS_MODEL, prompt, TaskProfile)

def build_judge(analysis, rubric):
    # Validate the rubric: it must score a gold answer HIGH and an empty answer LOW.
    # If it doesn't discriminate, blank it and fall back to the generic judge.
    golds = [e for e in analysis.answer_examples if str(e).strip()][:2]
    if not golds:
        return rubric, "no golds to validate; using rubric as-is"
    highs = []
    lows = []
    for gold in golds:
        highs.append(judge_score(gold, gold, rubric, analysis.description))  # gold vs itself -> high
        lows.append(judge_score("", gold, rubric, analysis.description))     # empty answer   -> low
    hi = sum(highs) / len(highs)
    lo = sum(lows) / len(lows)
    if hi >= 0.7 and lo <= 0.5:
        return rubric, "ok"
    return "", f"rubric didn't discriminate (gold={hi:.2f}, empty={lo:.2f}); using generic judge"

def plan_facets(analysis, k=15):
    # Ask for diverse sub-types / scenarios up front so the dataset spans them,
    # instead of every batch re-covering the same "typical" cases.
    prompt = (f"List {k} distinct facets / scenarios / sub-types that a diverse test set for "
              f"this task should span. Short phrases.\n{analysis.description}")
    try:
        return generate_json(ANALYSIS_MODEL, prompt, Facets).facets
    except Exception:
        return []

def _norm_question(text):
    # Normalized key for near-duplicate detection: lowercase, collapse any run of
    # non-alphanumeric chars to one space. Catches paraphrases exact-match misses.
    return re.sub(r"[^a-z0-9]+", " ", text.lower()).strip()

def generate_dataset(analysis, n=N_DATASET):
    # Generate in SMALL BATCHES — one giant structured-output call runs past
    # MAX_OUTPUT_TOKENS and comes back as truncated, invalid JSON. Free-form
    # answers are long, so use a smaller batch for those.
    #
    # Cross-batch diversity: each call is independent (the model has no memory of
    # earlier batches), so we (1) rotate batches across a set of facets, (2) show
    # each batch a few already-generated inputs to avoid, and (3) dedup on a
    # normalized key so paraphrases don't slip through.
    if analysis.check_type == "llm_judge":
        answer_rule = ("Each 'answer' is an ideal reference output for that input — it may be "
                       "multi-sentence / free-form; it will be graded by an LLM judge.")
        batch = 5
    else:
        answer_rule = ("Each 'answer' must be the correct final target ONLY — a bare value "
                       "(the number or the label), with no explanation or units.")
        batch = 20

    facets = plan_facets(analysis)
    print(f"generating ~{n} examples across {len(facets)} facets (batches of {batch})...")

    seen = set()          # normalized question keys
    data = []
    stalls = 0
    facet_i = 0
    while len(data) < n and stalls < 3:
        k = min(batch, n - len(data))

        facet_hint = ""
        if facets:
            chosen = [facets[(facet_i + j) % len(facets)] for j in range(3)]
            facet_i += 3
            facet_hint = " Cover these facets specifically: " + "; ".join(chosen) + "."

        avoid_hint = ""
        if data:
            recent = [item["question"][:80].replace("\n", " ") for item in data[-8:]]
            avoid_hint = ("\n\nMake them DIFFERENT from these already-generated inputs:\n- "
                          + "\n- ".join(recent))

        prompt = (f"Generate {k} diverse labeled examples for this task, different from "
                  f"typical / obvious cases:\n{analysis.description}\n\n"
                  + answer_rule + facet_hint + avoid_hint
                  + " Make them solvable and unambiguous.")
        try:
            batch_data = generate_json(ANALYSIS_MODEL, prompt, Dataset).data
        except Exception:
            batch_data = []              # truncated / garbled batch -> skip, don't crash
        before = len(data)
        for item in batch_data:
            key = _norm_question(item.question)
            if key and key not in seen:
                seen.add(key)
                # plain dicts, so generated and user-supplied DATA look the same
                data.append({"question": item.question, "answer": item.answer})
        stalls = stalls + 1 if len(data) == before else 0
    return data[:n]

analysis = analyze_task(SEED_PROMPT, DATASET)
CHECK_TYPE = analysis.check_type

JUDGE_TASK = analysis.description
if CHECK_TYPE == "llm_judge":
    JUDGE_RUBRIC, judge_status = build_judge(analysis, analysis.judge_rubric)
else:
    JUDGE_RUBRIC, judge_status = "", "n/a (not an LLM judge)"

if DATASET is not None:
    DATA = DATASET
else:
    DATA = generate_dataset(analysis)

# Split: the designer may self-test on DEV; the final ranking is on held-out TEST.
split = max(1, len(DATA) * 3 // 5)
DATA_DEV = DATA[:split]
DATA_TEST = DATA[split:]

print(f"check       = {CHECK_TYPE}")
if CHECK_TYPE == "llm_judge":
    print(f"judge       = {judge_status}")
    if JUDGE_RUBRIC:
        print(f"rubric      = {JUDGE_RUBRIC[:100]}{'...' if len(JUDGE_RUBRIC) > 100 else ''}")
print(f"answers     = {', '.join(repr(e) for e in analysis.answer_examples[:3])}")
print(f"{len(DATA)} examples  ->  {len(DATA_DEV)} dev / {len(DATA_TEST)} test")
if DATASET is None and len(DATA) < N_DATASET:
    print(f"(note: generated {len(DATA)}/{N_DATASET} unique examples)")
print("Description:", analysis.description)

**Examples — analysis.** `analyze_task` returns a validated `TaskProfile`; `build_judge` checks its rubric (or falls back); `generate_dataset` makes labeled examples.

```python
analyze_task("Grade-school math; answers are integers.", None)
# -> TaskProfile(description='...', check_type='numeric', judge_rubric='',
#                answer_examples=['9', '42', '156'])

analysis.check_type                              # typed attribute access, not analysis["check_type"]
TaskProfile.model_json_schema()                  # the JSON Schema sent to the API

build_judge(analysis, analysis.judge_rubric)     # -> (rubric, "ok")  or  ("", "rubric didn't discriminate (...)")
generate_dataset(analysis, n=10)                 # -> [{"question": ..., "answer": ...}, ...]
```

## Helpers for picking a workflow

Three small functions used in the final step, operating on a list of result dicts (`{"name", "accuracy", "cost_per_query"}`):
- `pareto_front(results)` — keep only the **non-dominated** results (see the examples below).
- `best_under_budget(results, max_cost)` — the most accurate result costing no more than `max_cost`.
- `cheapest_above_accuracy(results, min_accuracy)` — the cheapest result reaching `min_accuracy`.

In [ ]:
# ---- Pareto frontier + picking a workflow under a constraint --------------
def cost_of(result):
    return result["cost_per_query"]

def pareto_front(results):
    # Keep a result only if no OTHER result is at least as accurate AND at least
    # as cheap (and strictly better on at least one of the two).
    front = []
    for r in results:
        dominated = False
        for other in results:
            if other is r:
                continue
            at_least_as_cheap    = other["cost_per_query"] <= r["cost_per_query"]
            at_least_as_accurate = other["accuracy"] >= r["accuracy"]
            strictly_better      = (other["cost_per_query"] < r["cost_per_query"]
                                    or other["accuracy"] > r["accuracy"])
            if at_least_as_cheap and at_least_as_accurate and strictly_better:
                dominated = True
                break
        if not dominated:
            front.append(r)
    front.sort(key=cost_of)          # cheapest first
    return front

def best_under_budget(results, max_cost):
    best = None
    for r in results:
        if r["cost_per_query"] <= max_cost:
            if best is None or r["accuracy"] > best["accuracy"]:
                best = r
    return best

def cheapest_above_accuracy(results, min_accuracy):
    cheapest = None
    for r in results:
        if r["accuracy"] >= min_accuracy:
            if cheapest is None or r["cost_per_query"] < cheapest["cost_per_query"]:
                cheapest = r
    return cheapest

**Examples — Pareto selection** (pure functions, no API):

```python
results = [
    {"name": "cheap_direct",     "accuracy": 0.70, "cost_per_query": 0.0002},
    {"name": "cot",              "accuracy": 0.90, "cost_per_query": 0.0010},
    {"name": "self_consistency", "accuracy": 0.92, "cost_per_query": 0.0050},
    {"name": "dominated",        "accuracy": 0.80, "cost_per_query": 0.0020},
]

pareto_front(results)                   # -> cheap_direct, cot, self_consistency
# "dominated" drops out: "cot" is both cheaper AND more accurate than it.
best_under_budget(results, 0.0015)      # -> cot   (most accurate costing < $0.0015/query)
cheapest_above_accuracy(results, 0.90)  # -> cot   (cheapest that reaches 0.90 accuracy)
```

## Step 2 · The design agent (called once per optimization round)

`run_design_round(round_num, context)` runs the **Claude Agent SDK** agent in a subprocess and returns the workflows it proposed. On round 1 it designs a diverse initial set; on later rounds `context` shows it the best workflows so far so it can design **cheaper** ones that keep accuracy. `summarize_archive(archive)` builds that context — the current DEV Pareto frontier plus the most-accurate workflow's code, as a base to improve on.

The agent's method + its dev evaluator are the two `SKILL.md` skills under `skills/` (`workflow-design`, `workflow-eval`), copied into its `.claude/skills/` each round. It runs in a separate process (`run_proposer.py`) so its async loop stays isolated from the notebook kernel.

> Requires `claude-agent-sdk` + `ANTHROPIC_API_KEY`; each round makes real API calls (spends money).

In [ ]:
# ---- The design agent, wrapped so we can call it each round ----------------
import tempfile, pathlib, shutil, subprocess, sys

PROPOSER_MODEL = "claude-opus-4-8"

def summarize_archive(archive):
    # Context the agent sees each round: the current best tradeoffs (DEV accuracy
    # + cost), plus the code of the most accurate workflow as a base to improve on.
    if not archive:
        return ""
    dev_results = []
    for p in archive:
        dev_results.append({"name": p["name"], "accuracy": p["dev_accuracy"],
                            "cost_per_query": p["dev_cost"]})
    lines = []
    for r in pareto_front(dev_results):
        lines.append(f"- {r['name']}: accuracy {r['accuracy']:.2f}, ${r['cost_per_query']:.5f}/query")

    most_accurate = archive[0]
    for p in archive:
        if p["dev_accuracy"] > most_accurate["dev_accuracy"]:
            most_accurate = p
    lines.append(f"\nMost accurate so far is '{most_accurate['name']}' "
                 f"(accuracy {most_accurate['dev_accuracy']:.2f}, "
                 f"${most_accurate['dev_cost']:.5f}/query). Its code:\n{most_accurate['code']}")
    return "\n".join(lines)

def run_design_round(round_num, context):
    agent_dir = pathlib.Path(tempfile.mkdtemp(prefix=f"proposer_r{round_num}_"))
    # task_spec the dev evaluator (eval_candidate.py) reads: the checker + judge
    # rubric, so the agent grades candidates exactly as this notebook does. There
    # is no extractor to ship — a program returns its own answer.
    (agent_dir / "task_spec.json").write_text(json.dumps({
        "description": analysis.description,
        "check": {"type": CHECK_TYPE, "task": analysis.description, "rubric": JUDGE_RUBRIC},
    }))
    (agent_dir / "dev_task.json").write_text(json.dumps(DATA_DEV[:5]))
    skills_dir = agent_dir / ".claude" / "skills"
    skills_dir.mkdir(parents=True)
    for skill_name in ("workflow-design", "workflow-eval"):
        shutil.copytree(pathlib.Path("skills") / skill_name, skills_dir / skill_name)

    # What the agent needs to return an answer in the right SHAPE, not just the
    # right value — the checker is strict now, so "42 apples" scores 0 on a
    # numeric task. The examples come from the analyzer.
    facts = ("Available models, cheap -> expensive: " + ", ".join(MODELS) + ".\n"
             "solve() must RETURN its final answer, scored by check='" + CHECK_TYPE
             + "'. Nothing is parsed out of prose.")
    if CHECK_TYPE == "numeric":
        facts += " A numeric answer must BE the number, with no words or units around it."
    elif CHECK_TYPE == "exact":
        facts += " It is compared against the gold answer exactly, ignoring case."
    if analysis.answer_examples:
        facts += ("\nCorrectly formatted answers for this task look exactly like:\n- "
                  + "\n- ".join(str(e) for e in analysis.answer_examples[:4]))

    if round_num == 1:
        goal = ("Design 4-5 DIVERSE, working candidate workflows for this task, spanning "
                "cheap -> accurate:\n" + analysis.description)
    else:
        goal = ("Improve on the workflows found so far for this task:\n" + analysis.description
                + "\n\nWorkflows so far:\n" + context
                + "\n\nDesign 3-4 NEW workflows that keep accuracy at least as high as the best "
                "one above but cost LESS per query — e.g. a cheaper model, fewer calls, routing "
                "easy inputs to the cheap model, or code execution instead of many samples.")
    prompt = (goal + "\n\n" + facts + "\n\n"
              "Use the workflow-design skill for the contract and the workflow-eval skill to test "
              "each candidate on the dev set. Write your final picks to programs.json and stop.")

    (agent_dir / "proposer_config.json").write_text(json.dumps({
        "model": PROPOSER_MODEL,
        "skills": ["workflow-design", "workflow-eval"],
        "allowed_tools": ["WebSearch", "WebFetch", "Bash", "Read", "Write", "Skill"],
        "prompt": prompt,
    }))

    runner = str(pathlib.Path("run_proposer.py").resolve())
    process = subprocess.Popen([sys.executable, runner], cwd=str(agent_dir),
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    programs_file = agent_dir / "programs.json"
    if programs_file.exists():
        data = json.loads(programs_file.read_text())
        return data["programs"] if isinstance(data, dict) else data
    # salvage candidate .py files if programs.json is missing
    programs = []
    for f in sorted(agent_dir.glob("*.py")):
        if f.name in ("run_proposer.py", "eval_candidate.py"):
            continue
        code = f.read_text()
        if "def solve" in code:
            programs.append({"name": f.stem, "description": "(recovered)", "code": code})
    return programs

## The runtime that executes a candidate program

The machinery that actually *runs* a program:

- `Reply` — what `call_model` returns. It **is** a string (so `return call_model(p)` works unchanged), but carries the whole response: `.blocks` (every content block, including tool calls and their results), `.data` (parsed JSON when the call passed a `schema`), `.usage`, `.model`.
- `Runtime` — a per-query object. Its `.call_model(...)` method is the metered `call_model` a program receives: it counts tokens, adds up cost, and raises if the program exceeds its budget (max calls / tokens). It also appends every call to `.turns` as a `Turn`, so a run can be inspected afterwards.
- `compile_solve(code)` — runs a program's source and returns its `solve` function.
- `final_answer(returned)` — reduces what `solve()` returned to the string we grade: the value itself, or `returned["answer"]` if it handed back a dict.
- `evaluate_program(...)` — runs the program over a dataset and returns its mean score and mean cost, plus `records`: per example the answer, score, cost, error, and the **full trace of every model call**. The trace is kept for inspection and never graded.

In [ ]:
# ---- Run a workflow program, metering + capping every model call ----------
from collections import Counter

class Reply(str):
    """What call_model returns: the reply text, with the whole response attached.

    It IS a string, so `return call_model(prompt)` keeps working — but nothing is
    thrown away. `.blocks` holds every content block the call produced (tool calls,
    tool results, text), `.data` is the parsed JSON when the call passed a schema,
    and `.usage` / `.model` say what it cost.
    """
    def __new__(cls, text, blocks=(), usage=None, model="", data=None):
        reply = super().__new__(cls, text)
        reply.blocks = list(blocks)
        reply.usage = dict(usage or {})
        reply.model = model
        reply.data = data
        return reply

@dataclass
class Turn:
    """One metered model call, kept whole so the trace loses nothing."""
    model: str
    prompt: str
    reply: Reply
    cost: float

class Runtime:
    """A workflow program calls `runtime.call_model(...)` for every model call. This is
    the one place cost is measured, and it stops the program if it goes over budget.
    It also records every call in `.turns`, so a run can be inspected afterwards."""
    def __init__(self, default_model, max_calls=24, token_budget=120_000):
        self.default_model = default_model
        self.max_calls = max_calls
        self.token_budget = token_budget
        self.calls = 0
        self.tokens = 0
        self.cost = 0.0
        self.turns = []              # every call this query made, in order

    def call_model(self, prompt, model=None, system=None, tools=None,
                   effort=None, schema=None):
        if self.calls >= self.max_calls:
            raise RuntimeError("workflow exceeded its model-call budget")
        self.calls += 1
        if model not in BY_ID:
            model = self.default_model
        text, blocks, usage = _call_api(
            model, str(prompt), system=system, tools=tools, effort=effort, schema=schema)

        data = None
        if schema:
            try:
                data = json.loads(text)     # the reply is constrained to the schema
            except ValueError:
                data = None                 # a refusal, or a reply past the ceiling

        call_cost = cost_usd(model, usage)
        self.tokens += usage["input"] + usage["output"] + usage["cache_write"] + usage["cache_read"]
        self.cost += call_cost

        reply = Reply(text, blocks=blocks, usage=usage, model=model, data=data)
        self.turns.append(Turn(model=model, prompt=str(prompt), reply=reply, cost=call_cost))

        if self.tokens > self.token_budget:
            raise RuntimeError("workflow exceeded its token budget")
        return reply

# The shape of a final answer. Handed to every program so the graded contract and
# the schema it asks the model for cannot drift apart. This is the shape for the
# value solve() RETURNS — intermediate calls (a difficulty router, a decomposer)
# should use whatever schema fits them.
class Answer(BaseModel):
    model_config = ConfigDict(extra="forbid")
    answer: str

# Handed to programs as a plain dict: model-written code runs in a sandbox with no
# pydantic, and call_model's schema= accepts either form.
ANSWER_SCHEMA = Answer.model_json_schema()

def compile_solve(code):
    # Run the program's source and return its solve() function. The program may
    # use these names without importing them. (Model-written code runs with full
    # Python here — fine for a trusted research notebook; sandbox it for untrusted code.)
    namespace = {"re": re, "json": json, "statistics": statistics,
                 "Counter": Counter, "extract_last_number": extract_last_number,
                 "MODELS": MODELS, "ANSWER_SCHEMA": ANSWER_SCHEMA}
    exec(code, namespace)
    return namespace["solve"]

def final_answer(returned):
    # What solve() handed back, reduced to the string we grade. A program returns
    # its answer directly, a dict carrying the answer plus anything else it wants
    # to keep, or a schema-constrained Reply as-is — all three are unwrapped to the
    # answer. A structure with no "answer" in it is a contract violation, not an
    # answer: stringifying it would grade `{'result': 'positive'}` as the
    # prediction and silently score 0, so raise and let the record show why.
    if isinstance(returned, Reply) and returned.data is not None:
        returned = returned.data
    if isinstance(returned, dict):
        if "answer" not in returned:
            raise ValueError(
                f"solve() returned an object with no 'answer' key: {sorted(returned)}")
        returned = returned["answer"]
    return str(returned).strip()

def evaluate_program(program, dataset, default_model):
    # Run the program once per example: grade the answer it returns, tally what its
    # model calls cost, and keep the full trace of every call. A program that
    # crashes or blows its budget scores 0 on that example rather than sinking the
    # whole evaluation.
    try:
        solve = compile_solve(program["code"])
    except Exception as error:
        return {"name": program["name"], "accuracy": 0.0, "cost_per_query": 0.0,
                "records": [], "error": f"compile: {error}"}

    records = []
    for item in dataset:
        runtime = Runtime(default_model)
        answer, score, error = "", 0.0, None
        try:
            returned = solve(item["question"], runtime.call_model)
            answer = final_answer(returned)
            score = check_answer(answer, item["answer"], CHECK_TYPE)
        except Exception as e:
            error = f"{type(e).__name__}: {e}"
        records.append({"question": item["question"], "gold": item["answer"],
                        "answer": answer, "score": score, "cost": runtime.cost,
                        "error": error,
                        "turns": runtime.turns})     # full trace: prompts, blocks, usage

    return {"name": program["name"],
            "accuracy": statistics.mean(r["score"] for r in records),   # mean score
            "cost_per_query": statistics.mean(r["cost"] for r in records),
            "records": records}                      # kept, never graded

**Examples — running a workflow program.** A program is source code defining `solve(question, call_model)` that **returns its answer**.

```python
code = '''
def solve(question, call_model):
    reply = call_model(question, schema={"type": "object",
                                         "properties": {"answer": {"type": "string"}},
                                         "required": ["answer"],
                                         "additionalProperties": False})
    return reply.data["answer"]        # .data is the parsed JSON
'''
solve = compile_solve(code)             # -> a function(question, call_model)

result = evaluate_program({"name": "direct", "code": code}, DATA_DEV, MODELS[0])
result["accuracy"], result["cost_per_query"]
result["records"][0]["answer"]          # what the program returned for example 0
result["records"][0]["turns"][0].reply.blocks   # every content block of its first call
```

A program can also skip the schema and just `return call_model(prompt)` — the reply is a string. Either way `solve` returns the answer; nothing is parsed out of prose.

## Step 3 · Optimize — loop the designer to cut cost without losing accuracy

Run the designer `N_ROUNDS` times. Each round it tries to beat the current best on **cost** while holding **accuracy**; every new candidate is scored on the **DEV** split and added to the `archive`. Over the rounds the archive fills with progressively more efficient workflows.

> This is the most expensive cell — it runs the agent `N_ROUNDS` times and scores every candidate on DEV, so it makes a lot of API calls. Lower `N_ROUNDS` (e.g. to 1) to spend less.

In [ ]:
# ---- The optimization loop -------------------------------------------------
N_ROUNDS = 3
BASE_MODEL = MODELS[0]     # a workflow's default model; it may escalate itself
archive = []               # every candidate seen: {name, description, code, dev_accuracy, dev_cost}

for round_num in range(1, N_ROUNDS + 1):
    print(f"\n===== design round {round_num} / {N_ROUNDS} =====")
    context = summarize_archive(archive)                       # empty on round 1
    for program in run_design_round(round_num, context):
        if any(program["code"] == seen["code"] for seen in archive):   # skip exact repeats
            continue
        scored = evaluate_program(program, DATA_DEV, BASE_MODEL)        # score on DEV
        program["dev_accuracy"] = scored["accuracy"]
        program["dev_cost"] = scored["cost_per_query"]
        program["dev_records"] = scored["records"]   # answers + full call traces
        archive.append(program)
        print(f"  + {program['name']:22s} dev acc {program['dev_accuracy']:.2f}  "
              f"${program['dev_cost']:.5f}/query")

print(f"\n{len(archive)} workflows in the archive after {N_ROUNDS} rounds.")

## Rank the finalists on held-out TEST

The DEV scores guided the loop; now we get honest numbers on the **TEST** split (which nothing was tuned against). To keep cost down we only TEST-evaluate the DEV Pareto frontier — the candidates actually worth choosing between. This produces `code_results`.

In [ ]:
# ---- Rank the finalists on the held-out TEST split -------------------------
# DEV scores guided the loop; now we report honest numbers on TEST (which nothing
# was tuned against). To keep cost down we only TEST-evaluate the DEV Pareto
# frontier — the candidates actually worth choosing between.
dev_results = []
for p in archive:
    dev_results.append({"name": p["name"], "accuracy": p["dev_accuracy"],
                        "cost_per_query": p["dev_cost"],
                        "code": p["code"], "description": p.get("description", "")})

finalists = pareto_front(dev_results)     # the non-dominated workflows on DEV

code_results = []
for p in finalists:
    scored = evaluate_program(p, DATA_TEST, BASE_MODEL)
    scored["description"] = p["description"]
    scored["code"] = p["code"]
    code_results.append(scored)

pd.DataFrame([{"name": r["name"], "accuracy": r["accuracy"],
               "cost_per_query": r["cost_per_query"]} for r in code_results]).sort_values(
    ["accuracy", "cost_per_query"], ascending=[False, True]).reset_index(drop=True)

## The tradeoffs — frontier + two automatic picks

The **Pareto frontier** over the finalists (cheapest → most accurate), plus two convenience picks — the best workflow under a dollar budget, and the cheapest above an accuracy floor — and a plot of every finalist with the frontier drawn through them.

In [ ]:
# ---- The Pareto frontier, the two constrained picks, and a plot ------------
frontier = pareto_front(code_results)

print("Pareto frontier (cheapest -> most accurate):")
for r in frontier:
    print(f"  {r['name']:22s}  accuracy {r['accuracy']:.2f}   ${r['cost_per_query']:.5f}/query")

BUDGET = 0.002
best = best_under_budget(code_results, BUDGET)
if best is None:
    print(f"\nNo program costs under ${BUDGET}/query.")
else:
    print(f"\nBest program under ${BUDGET}/query:  {best['name']}"
          f"  (accuracy {best['accuracy']:.2f}, ${best['cost_per_query']:.5f})")

TARGET = 0.80
cheapest = cheapest_above_accuracy(code_results, TARGET)
if cheapest is None:
    print(f"No program reaches accuracy {TARGET}.")
else:
    print(f"Cheapest program with accuracy >= {TARGET}:  {cheapest['name']}"
          f"  (accuracy {cheapest['accuracy']:.2f}, ${cheapest['cost_per_query']:.5f})")

# Plot every program as a point; draw a line through the frontier.
frontier_costs = []
frontier_accuracies = []
for r in frontier:
    frontier_costs.append(r["cost_per_query"])
    frontier_accuracies.append(r["accuracy"])

plt.figure(figsize=(7, 5))
for r in code_results:
    plt.scatter(r["cost_per_query"], r["accuracy"], color="#888888")
    plt.annotate(r["name"], (r["cost_per_query"], r["accuracy"]),
                 fontsize=8, xytext=(5, 4), textcoords="offset points")
plt.plot(frontier_costs, frontier_accuracies, "-o", color="#d9534f", label="Pareto frontier")
plt.xlabel("cost per query (USD)")
plt.ylabel("accuracy")
plt.title("LLM-designed workflows: accuracy vs. cost")
plt.legend()
plt.tight_layout()
plt.show()

## Step 4 · Choose your methodology

Every workflow below is on the frontier — a genuine option at a different accuracy/cost point. Read each one's code, then set `CHOICE` to the name you want; `chosen["code"]` is what you'd ship.

In [ ]:
# ---- Choose your methodology ----------------------------------------------
# Each workflow on the frontier is a real option — a different accuracy/cost
# point. Read them, then set CHOICE to the one you want; `chosen["code"]` is the
# workflow you'd ship.
for r in pareto_front(code_results):
    print("=" * 72)
    print(f"{r['name']}   —   accuracy {r['accuracy']:.2f},  ${r['cost_per_query']:.5f}/query")
    if r.get("description"):
        print(r["description"])
    print("-" * 72)
    print(r["code"].strip())
    print()

CHOICE = pareto_front(code_results)[-1]["name"]   # default: the most accurate frontier workflow
chosen = None
for r in code_results:
    if r["name"] == CHOICE:
        chosen = r

print("=" * 72)
print(f"CHOSEN: {CHOICE}   (edit CHOICE above to pick a different tradeoff)\n")
print(chosen["code"].strip())